### resolve csv.gz.icloud issue after downloading files from google drive using gdown
is a placeholder created by macOS iCloud Drive optimization. It means:  
- The actual file is in iCloud, not on your disk.  
- Python (or any app outside Finder) cannot read it until it’s fully downloaded.  

In [1]:
# import os
# print('-'*40, 'hosp', '-'*40)
# path = "/Users/ginger/Downloads/DSCI531_Project/dataset/mimic4/hosp"
# for i in os.listdir(path):
#     print(i)
# print('-'*40, 'icu', '-'*40)
# path = "/Users/ginger/Downloads/DSCI531_Project/dataset/mimic4/icu"
# for i in os.listdir(path):
#     print(i)

In [2]:
import pandas as pd

# Core demographic and admission data
patients = pd.read_csv("dataset/mimic4/hosp/patients.csv.gz")
admissions = pd.read_csv("dataset/mimic4/hosp/admissions.csv.gz")

# ICU stays
icustays = pd.read_csv("dataset/mimic4/icu/icustays.csv.gz")

# Diagnoses (optional but useful)
diagnoses = pd.read_csv("dataset/mimic4/hosp/diagnoses_icd.csv.gz")

cols = ['subject_id', 'hadm_id', 'stay_id', 'itemid', 'charttime', 'valuenum']
chartevents = pd.read_csv("dataset/mimic4/icu/chartevents.csv.gz", usecols=cols)

cols = ['subject_id', 'hadm_id', 'itemid', 'charttime', 'valuenum']                           
labevents = pd.read_csv("dataset/mimic4/hosp/labevents.csv.gz", usecols=cols)

# Dictionaries for interpretation
d_items = pd.read_csv("dataset/mimic4/icu/d_items.csv.gz")
d_labitems = pd.read_csv("dataset/mimic4/hosp/d_labitems.csv.gz")


## Preprocessing

### patiens

In [3]:
patients.head()

,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod
0,10000032,F,52,2180,2014 - 2016,2180-09-09
1,10000048,F,23,2126,2008 - 2010,NaN
2,10000058,F,33,2168,2020 - 2022,NaN
3,10000068,F,19,2160,2008 - 2010,NaN
4,10000084,M,72,2160,2017 - 2019,2161-02-13


In [4]:
print(f'''patients:
{patients.shape}
{'-'*100}
{patients.dtypes}
{'-'*100}
{patients.count()}
{'-'*100}
{patients.isna().sum()}
{'-'*100}
{patients.describe()}
{'-'*100}
''')

patients:
(364627, 6)
----------------------------------------------------------------------------------------------------
subject_id            int64
gender               object
anchor_age            int64
anchor_year           int64
anchor_year_group    object
dod                  object
dtype: object
----------------------------------------------------------------------------------------------------
subject_id           364627
gender               364627
anchor_age           364627
anchor_year          364627
anchor_year_group    364627
dod                   38301
dtype: int64
----------------------------------------------------------------------------------------------------
subject_id                0
gender                    0
anchor_age                0
anchor_year               0
anchor_year_group         0
dod                  326326
dtype: int64
----------------------------------------------------------------------------------------------------
         subject_id     anchor

In [5]:
# Drop synthetic time columns
patients_clean = patients.copy()
patients_clean = patients_clean.drop(columns=['anchor_year', 'anchor_year_group'])

# Convert gender to binary
patients_clean['gender'] = patients_clean['gender'].map({'M': 1, 'F': 0})

# Convert dod to datetime
patients_clean['dod'] = pd.to_datetime(patients_clean['dod'])

# dod will have many missing values — that’s expected (most patients survived)
print(f'''patients_clean:
{patients_clean.shape}
{'-'*100}
{patients_clean.count()}
{'-'*100}
{patients_clean.isna().sum()}
{'-'*100}
''')
patients_clean.info()
patients_clean.head()

patients_clean:
(364627, 4)
----------------------------------------------------------------------------------------------------
subject_id    364627
gender        364627
anchor_age    364627
dod            38301
dtype: int64
----------------------------------------------------------------------------------------------------
subject_id         0
gender             0
anchor_age         0
dod           326326
dtype: int64
----------------------------------------------------------------------------------------------------

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 364627 entries, 0 to 364626
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype         
---  ------      --------------   -----         
 0   subject_id  364627 non-null  int64         
 1   gender      364627 non-null  int64         
 2   anchor_age  364627 non-null  int64         
 3   dod         38301 non-null   datetime64[ns]
dtypes: datetime64[ns](1), int64(3)
memory usage: 11.1 MB


,subject_id,gender,anchor_age,dod
0,10000032,0,52,2180-09-09
1,10000048,0,23,NaT
2,10000058,0,33,NaT
3,10000068,0,19,NaT
4,10000084,1,72,2161-02-13


### admissions

In [6]:
admissions.head()

,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaN,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,English,WIDOWED,WHITE,2180-05-06 19:17:00,2180-05-06 23:30:00,0
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaN,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-06-26 15:54:00,2180-06-26 21:31:00,0
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaN,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,English,WIDOWED,WHITE,2180-08-05 20:58:00,2180-08-06 01:44:00,0
3,10000032,29079034,2180-07-23 12:35:00,2180-07-25 17:55:00,NaN,EW EMER.,P06OTX,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-07-23 05:54:00,2180-07-23 14:00:00,0
4,10000068,25022803,2160-03-03 23:16:00,2160-03-04 06:26:00,NaN,EU OBSERVATION,P39NWO,EMERGENCY ROOM,NaN,NaN,English,SINGLE,WHITE,2160-03-03 21:55:00,2160-03-04 06:26:00,0


In [7]:
print(f'''admissions:
{admissions.shape}
{'-'*100}
{admissions.dtypes}
{'-'*100}
{admissions.count()}
{'-'*100}
{admissions.isna().sum()}
{'-'*100}
{admissions.describe()}
{'-'*100}
''')

admissions:
(546028, 16)
----------------------------------------------------------------------------------------------------
subject_id               int64
hadm_id                  int64
admittime               object
dischtime               object
deathtime               object
admission_type          object
admit_provider_id       object
admission_location      object
discharge_location      object
insurance               object
language                object
marital_status          object
race                    object
edregtime               object
edouttime               object
hospital_expire_flag     int64
dtype: object
----------------------------------------------------------------------------------------------------
subject_id              546028
hadm_id                 546028
admittime               546028
dischtime               546028
deathtime                11790
admission_type          546028
admit_provider_id       546024
admission_location      546027
discharge_locat

In [8]:
cols = ['subject_id', 'hadm_id', 'dischtime', 'insurance', 'language', 'marital_status', 'race', 'hospital_expire_flag']

admissions_clean = admissions[cols].copy()

# Convert datetime columns
admissions_clean['dischtime'] = pd.to_datetime(admissions_clean['dischtime'])

# Fill missing values for sensitive attributes
'''“This particular NaN value has special meaning — it tells us that the information was unknown or missing at the time.
Instead of dropping or guessing it, we treat it as its own category by converting it into a label like 'UNKNOWN', so the model can process it.”'''

admissions_clean[['insurance', 'language', 'marital_status']] = admissions_clean[['insurance', 'language', 'marital_status']].fillna('UNKNOWN')


# One-hot
# admissions_clean['race'].value_counts()

def simplify_race(race):
    race = str(race).upper().strip()

    if "WHITE" in race:
        return "WHITE"
    elif "BLACK" in race:
        return "BLACK"
    elif "ASIAN" in race:
        return "ASIAN"
    elif "HISPANIC" in race or "LATINO" in race:
        return "HISPANIC/LATINO"
    elif "PACIFIC ISLANDER" in race or "HAWAIIAN" in race:
        return "PACIFIC ISLANDER"
    elif "AMERICAN INDIAN" in race or "ALASKA NATIVE" in race:
        return "AMERICAN INDIAN/ALASKA NATIVE"
    elif race in ["OTHER", "PORTUGUESE", "SOUTH AMERICAN"]:
        return "OTHER"
    elif race in ["UNKNOWN", "UNABLE TO OBTAIN", "PATIENT DECLINED TO ANSWER", "MULTIPLE RACE/ETHNICITY"]:
        return "UNKNOWN/DECLINED"
    else:
        return "OTHER"

admissions_clean['race_grouped'] = admissions_clean['race'].apply(simplify_race)
admissions_clean['race_grouped'].value_counts()


admissions_clean.drop(columns=['race', 'insurance', 'language', 'marital_status'], inplace=True)


# Preview cleaned data
print(f'''admissions_clean:
{admissions_clean.shape}
{'-'*100}
{admissions_clean.count()}
{'-'*100}
{admissions_clean.isna().sum()}
{'-'*100}
''')
admissions_clean.info()
admissions_clean.head()

admissions_clean:
(546028, 5)
----------------------------------------------------------------------------------------------------
subject_id              546028
hadm_id                 546028
dischtime               546028
hospital_expire_flag    546028
race_grouped            546028
dtype: int64
----------------------------------------------------------------------------------------------------
subject_id              0
hadm_id                 0
dischtime               0
hospital_expire_flag    0
race_grouped            0
dtype: int64
----------------------------------------------------------------------------------------------------

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 546028 entries, 0 to 546027
Data columns (total 5 columns):
 #   Column                Non-Null Count   Dtype         
---  ------                --------------   -----         
 0   subject_id            546028 non-null  int64         
 1   hadm_id               546028 non-null  int64         
 2   disc

,subject_id,hadm_id,dischtime,hospital_expire_flag,race_grouped
0,10000032,22595853,2180-05-07 17:15:00,0,WHITE
1,10000032,22841357,2180-06-27 18:49:00,0,WHITE
2,10000032,25742920,2180-08-07 17:50:00,0,WHITE
3,10000032,29079034,2180-07-25 17:55:00,0,WHITE
4,10000068,25022803,2160-03-04 06:26:00,0,WHITE


### icustays

In [9]:
icustays.head()

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535
3,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032
4,10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113


In [10]:
print(f'''icustays:
{icustays.shape}
{'-'*100}
{icustays.dtypes}
{'-'*100}
{icustays.count()}
{'-'*100}
{icustays.isna().sum()}
{'-'*100}
{icustays.describe()}
{'-'*100}
''')

icustays:
(94458, 8)
----------------------------------------------------------------------------------------------------
subject_id          int64
hadm_id             int64
stay_id             int64
first_careunit     object
last_careunit      object
intime             object
outtime            object
los               float64
dtype: object
----------------------------------------------------------------------------------------------------
subject_id        94458
hadm_id           94458
stay_id           94458
first_careunit    94458
last_careunit     94458
intime            94458
outtime           94444
los               94444
dtype: int64
----------------------------------------------------------------------------------------------------
subject_id         0
hadm_id            0
stay_id            0
first_careunit     0
last_careunit      0
intime             0
outtime           14
los               14
dtype: int64
--------------------------------------------------------------------

In [11]:
'''Don't use los for ICU mortality prediction because it creates a leakage risk.
If someone dies early, their los will be very short → model might learn to predict death from low los directly.
You want to predict it based on vitals, labs, and demographics available early, so use los for descriptive statistics and filtering out noise'''


# You can’t define ICU mortality without outtime so drop those missing rows
# Drops entire rows where the column outtime is NaN
icustays_clean = icustays.copy()
icustays_clean = icustays_clean.dropna(subset=['outtime'])

# Convert intime and outtime to datetime
icustays_clean['intime'] = pd.to_datetime(icustays_clean['intime'])
icustays_clean['outtime'] = pd.to_datetime(icustays_clean['outtime'])

cols = ['subject_id', 'hadm_id', 'stay_id', 'intime', 'outtime', 'los']
icustays_clean = icustays_clean[cols]

# Preview cleaned data
print(f'''icustays_clean:
{icustays_clean.shape}
{'-'*100}
{icustays_clean.count()}
{'-'*100}
{icustays_clean.isna().sum()}
{'-'*100}
''')
icustays_clean.info()
icustays_clean.head()

icustays_clean:
(94444, 6)
----------------------------------------------------------------------------------------------------
subject_id    94444
hadm_id       94444
stay_id       94444
intime        94444
outtime       94444
los           94444
dtype: int64
----------------------------------------------------------------------------------------------------
subject_id    0
hadm_id       0
stay_id       0
intime        0
outtime       0
los           0
dtype: int64
----------------------------------------------------------------------------------------------------

<class 'pandas.core.frame.DataFrame'>
Index: 94444 entries, 0 to 94457
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   subject_id  94444 non-null  int64         
 1   hadm_id     94444 non-null  int64         
 2   stay_id     94444 non-null  int64         
 3   intime      94444 non-null  datetime64[ns]
 4   outtime     94444 non-null  dat

,subject_id,hadm_id,stay_id,intime,outtime,los
0,10000032,29079034,39553978,2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266
1,10000690,25860671,37081114,2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252
2,10000980,26913865,39765666,2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535
3,10001217,24597018,37067082,2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032
4,10001217,27703517,34592300,2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113


### diagnoses

In [12]:
diagnoses.head()

,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10000032,22595853,1,5723,9
1,10000032,22595853,2,78959,9
2,10000032,22595853,3,5715,9
3,10000032,22595853,4,07070,9
4,10000032,22595853,5,496,9


In [13]:
print(f'''diagnoses:
{diagnoses.shape}
{'-'*100}
{diagnoses.dtypes}
{'-'*100}
{diagnoses.count()}
{'-'*100}
{diagnoses.isna().sum()}
{'-'*100}
{diagnoses.describe()}
{'-'*100}
''')

diagnoses:
(6364488, 5)
----------------------------------------------------------------------------------------------------
subject_id      int64
hadm_id         int64
seq_num         int64
icd_code       object
icd_version     int64
dtype: object
----------------------------------------------------------------------------------------------------
subject_id     6364488
hadm_id        6364488
seq_num        6364488
icd_code       6364488
icd_version    6364488
dtype: int64
----------------------------------------------------------------------------------------------------
subject_id     0
hadm_id        0
seq_num        0
icd_code       0
icd_version    0
dtype: int64
----------------------------------------------------------------------------------------------------
         subject_id       hadm_id       seq_num   icd_version
count  6.364488e+06  6.364488e+06  6.364488e+06  6.364488e+06
mean   1.500237e+07  2.500059e+07  8.825529e+00  9.542973e+00
std    2.878400e+06  2.889094e+06  6

In [14]:
# Charlson/Elixhauser handle ICD-9 only — pre-filter ICD-9 to exclude ICD-10
diagnoses_clean = diagnoses[diagnoses['icd_version'] == 9].copy()

cols = ['subject_id', 'hadm_id', 'icd_code']
diagnoses_clean = diagnoses_clean[cols]

# Convert icd_code to string and standardize format
diagnoses_clean = diagnoses_clean[cols]
diagnoses_clean['icd_code'] = diagnoses_clean['icd_code'].astype(str).str.strip().str.upper()

# Preview cleaned data
print(f'''diagnoses_clean:
{diagnoses_clean.shape}
{'-'*100}
{diagnoses_clean.count()}
{'-'*100}
{diagnoses_clean.isna().sum()}
{'-'*100}
''')
diagnoses_clean.info()
diagnoses_clean.head()

diagnoses_clean:
(2908741, 3)
----------------------------------------------------------------------------------------------------
subject_id    2908741
hadm_id       2908741
icd_code      2908741
dtype: int64
----------------------------------------------------------------------------------------------------
subject_id    0
hadm_id       0
icd_code      0
dtype: int64
----------------------------------------------------------------------------------------------------

<class 'pandas.core.frame.DataFrame'>
Index: 2908741 entries, 0 to 6364487
Data columns (total 3 columns):
 #   Column      Dtype 
---  ------      ----- 
 0   subject_id  int64 
 1   hadm_id     int64 
 2   icd_code    object
dtypes: int64(2), object(1)
memory usage: 88.8+ MB


,subject_id,hadm_id,icd_code
0,10000032,22595853,5723
1,10000032,22595853,78959
2,10000032,22595853,5715
3,10000032,22595853,07070
4,10000032,22595853,496


### chartevents

In [15]:
chartevents.head()

,subject_id,hadm_id,stay_id,charttime,itemid,valuenum
0,10000032,29079034,39553978,2180-07-23 12:36:00,226512,39.4
1,10000032,29079034,39553978,2180-07-23 12:36:00,226707,60.0
2,10000032,29079034,39553978,2180-07-23 12:36:00,226730,152.0
3,10000032,29079034,39553978,2180-07-23 14:00:00,220048,NaN
4,10000032,29079034,39553978,2180-07-23 14:00:00,224642,NaN


In [16]:
print(f'''chartevents:
{chartevents.shape}
{'-'*100}
{chartevents.dtypes}
{'-'*100}
{chartevents.count()}
{'-'*100}
{chartevents.isna().sum()}
{'-'*100}
{chartevents.describe()}
{'-'*100}
''')

chartevents:
(432997491, 6)
----------------------------------------------------------------------------------------------------
subject_id      int64
hadm_id         int64
stay_id         int64
charttime      object
itemid          int64
valuenum      float64
dtype: object
----------------------------------------------------------------------------------------------------
subject_id    432997491
hadm_id       432997491
stay_id       432997491
charttime     432997491
itemid        432997491
valuenum      169211246
dtype: int64
----------------------------------------------------------------------------------------------------
subject_id            0
hadm_id               0
stay_id               0
charttime             0
itemid                0
valuenum      263786245
dtype: int64
----------------------------------------------------------------------------------------------------
         subject_id       hadm_id       stay_id        itemid      valuenum
count  4.329975e+08  4.329975e+0

In [37]:
# Drop rows where the measurement is missing
'''Over 400 million rows. But only ~169 million rows have numeric values in valuenum
The rest (263M) are textual observations, equipment notes, categories, etc.'''
chartevents_clean = chartevents.dropna(subset=['valuenum']).copy()

# top 5 most frequent itemid values
top5 = chartevents_clean['itemid'].value_counts().head(5).index.tolist()
chartevents_clean = chartevents_clean[chartevents_clean['itemid'].isin(top5)].copy()

# Preview cleaned data
print(f'''chartevents_clean:
{chartevents_clean.shape}
{'-'*100}
{chartevents_clean.count()}
{'-'*100}
{chartevents_clean.isna().sum()}
{'-'*100}
''')
chartevents_clean.info()
chartevents_clean.head()

chartevents_clean:
(36712168, 6)
----------------------------------------------------------------------------------------------------
subject_id    36712168
hadm_id       36712168
stay_id       36712168
charttime     36712168
itemid        36712168
valuenum      36712168
dtype: int64
----------------------------------------------------------------------------------------------------
subject_id    0
hadm_id       0
stay_id       0
charttime     0
itemid        0
valuenum      0
dtype: int64
----------------------------------------------------------------------------------------------------

<class 'pandas.core.frame.DataFrame'>
Index: 36712168 entries, 7 to 432997414
Data columns (total 6 columns):
 #   Column      Dtype  
---  ------      -----  
 0   subject_id  int64  
 1   hadm_id     int64  
 2   stay_id     int64  
 3   charttime   object 
 4   itemid      int64  
 5   valuenum    float64
dtypes: float64(1), int64(4), object(1)
memory usage: 1.9+ GB


,subject_id,hadm_id,stay_id,charttime,itemid,valuenum
7,10000032,29079034,39553978,2180-07-23 14:11:00,220179,84.0
8,10000032,29079034,39553978,2180-07-23 14:11:00,220180,48.0
10,10000032,29079034,39553978,2180-07-23 14:12:00,220045,91.0
11,10000032,29079034,39553978,2180-07-23 14:12:00,220210,24.0
12,10000032,29079034,39553978,2180-07-23 14:13:00,220277,98.0


### labevents

In [18]:
labevents.head()

,subject_id,hadm_id,itemid,charttime,valuenum
0,10000032,NaN,50931,2180-03-23 11:51:00,95.0
1,10000032,NaN,51071,2180-03-23 11:51:00,NaN
2,10000032,NaN,51074,2180-03-23 11:51:00,NaN
3,10000032,NaN,51075,2180-03-23 11:51:00,NaN
4,10000032,NaN,51079,2180-03-23 11:51:00,NaN


In [19]:
print(f'''labevents:
{labevents.shape}
{'-'*100}
{labevents.dtypes}
{'-'*100}
{labevents.count()}
{'-'*100}
{labevents.isna().sum()}
{'-'*100}
{labevents.describe()}
{'-'*100}
''')

labevents:
(158374764, 5)
----------------------------------------------------------------------------------------------------
subject_id      int64
hadm_id       float64
itemid          int64
charttime      object
valuenum      float64
dtype: object
----------------------------------------------------------------------------------------------------
subject_id    158374764
hadm_id        84605867
itemid        158374764
charttime     158374764
valuenum      136884423
dtype: int64
----------------------------------------------------------------------------------------------------
subject_id           0
hadm_id       73768897
itemid               0
charttime            0
valuenum      21490341
dtype: int64
----------------------------------------------------------------------------------------------------
         subject_id       hadm_id        itemid      valuenum
count  1.583748e+08  8.460587e+07  1.583748e+08  1.368844e+08
mean   1.501474e+07  2.500166e+07  5.117165e+04  6.882403e+01

In [ ]:
# Drop rows without numeric values (non-numeric test results)
labevents_clean = labevents.dropna(subset=['valuenum']).copy()

# top 5 most frequent lab itemid values
top5 = labevents_clean['itemid'].value_counts().head(5).index.tolist()
labevents_clean = labevents_clean[labevents_clean['itemid'].isin(top5)].copy()

# Preview cleaned data
print(f'''labevents_clean:
{labevents_clean.shape}
{'-'*100}
{labevents_clean.count()}
{'-'*100}
{labevents_clean.isna().sum()}
{'-'*100}
''')
labevents_clean.info()
labevents_clean.head()

labevents_clean:
(21209243, 5)
----------------------------------------------------------------------------------------------------
subject_id    21209243
hadm_id       12621837
itemid        21209243
charttime     21209243
valuenum      21209243
dtype: int64
----------------------------------------------------------------------------------------------------
subject_id          0
hadm_id       8587406
itemid              0
charttime           0
valuenum            0
dtype: int64
----------------------------------------------------------------------------------------------------

<class 'pandas.core.frame.DataFrame'>
Index: 21209243 entries, 25 to 158374763
Data columns (total 5 columns):
 #   Column      Dtype  
---  ------      -----  
 0   subject_id  int64  
 1   hadm_id     float64
 2   itemid      int64  
 3   charttime   object 
 4   valuenum    float64
dtypes: float64(2), int64(2), object(1)
memory usage: 970.9+ MB


,subject_id,hadm_id,itemid,charttime,valuenum
25,10000032,NaN,50912,2180-03-23 11:51:00,0.4
45,10000032,NaN,51006,2180-03-23 11:51:00,13.0
50,10000032,NaN,51221,2180-03-23 11:51:00,45.4
51,10000032,NaN,51222,2180-03-23 11:51:00,14.9
58,10000032,NaN,51265,2180-03-23 11:51:00,83.0


### d_items

In [21]:
d_items.head()

,itemid,label,abbreviation,linksto,category,unitname,param_type,lownormalvalue,highnormalvalue
0,220001,Problem List,Problem List,chartevents,General,NaN,Text,NaN,NaN
1,220003,ICU Admission date,ICU Admission date,datetimeevents,ADT,NaN,Date and time,NaN,NaN
2,220045,Heart Rate,HR,chartevents,Routine Vital Signs,bpm,Numeric,NaN,NaN
3,220046,Heart rate Alarm - High,HR Alarm - High,chartevents,Alarms,bpm,Numeric,NaN,NaN
4,220047,Heart Rate Alarm - Low,HR Alarm - Low,chartevents,Alarms,bpm,Numeric,NaN,NaN


In [22]:
print(f'''d_items:
{d_items.shape}
{'-'*100}
{d_items.dtypes}
{'-'*100}
{d_items.count()}
{'-'*100}
{d_items.isna().sum()}
{'-'*100}
{d_items.describe()}
{'-'*100}
''')

d_items:
(4095, 9)
----------------------------------------------------------------------------------------------------
itemid               int64
label               object
abbreviation        object
linksto             object
category            object
unitname            object
param_type          object
lownormalvalue     float64
highnormalvalue    float64
dtype: object
----------------------------------------------------------------------------------------------------
itemid             4095
label              4095
abbreviation       4095
linksto            4095
category           4095
unitname           1123
param_type         4095
lownormalvalue       19
highnormalvalue      22
dtype: int64
----------------------------------------------------------------------------------------------------
itemid                0
label                 0
abbreviation          0
linksto               0
category              0
unitname           2972
param_type            0
lownormalvalue     4076


In [23]:
# Filter only rows that belong to chartevents
d_items_clean = d_items.copy()
d_items_clean = d_items_clean[d_items_clean['linksto'] == 'chartevents']

# Create a binary flag: is_vital_sign
# for category, count in d_items_clean['category'].value_counts().items():
#     print(category, count)

d_items_clean['is_vital_sign'] = (d_items_clean['category'] == 'Routine Vital Signs').map({True: 1, False: 0})

# drop linksto, category after filtering and other unneeded columns
cols = ['itemid', 'label', 'is_vital_sign']
d_items_clean = d_items_clean[cols]

# Preview cleaned data
print(f'''d_items_clean:
{d_items_clean.shape}
{'-'*100}
{d_items_clean.count()}
{'-'*100}
{d_items_clean.isna().sum()}
{'-'*100}
''')
d_items_clean.info()
d_items_clean.head()

d_items_clean:
(3055, 3)
----------------------------------------------------------------------------------------------------
itemid           3055
label            3055
is_vital_sign    3055
dtype: int64
----------------------------------------------------------------------------------------------------
itemid           0
label            0
is_vital_sign    0
dtype: int64
----------------------------------------------------------------------------------------------------

<class 'pandas.core.frame.DataFrame'>
Index: 3055 entries, 0 to 4094
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   itemid         3055 non-null   int64 
 1   label          3055 non-null   object
 2   is_vital_sign  3055 non-null   int64 
dtypes: int64(2), object(1)
memory usage: 95.5+ KB


,itemid,label,is_vital_sign
0,220001,Problem List,0
2,220045,Heart Rate,1
3,220046,Heart rate Alarm - High,0
4,220047,Heart Rate Alarm - Low,0
5,220048,Heart Rhythm,1


### d_labitems

In [24]:
d_labitems.head()

,itemid,label,fluid,category
0,50801,Alveolar-arterial Gradient,Blood,Blood Gas
1,50802,Base Excess,Blood,Blood Gas
2,50803,"Calculated Bicarbonate, Whole Blood",Blood,Blood Gas
3,50804,Calculated Total CO2,Blood,Blood Gas
4,50805,Carboxyhemoglobin,Blood,Blood Gas


In [25]:
print(f'''d_labitems:
{d_labitems.shape}
{'-'*100}
{d_labitems.dtypes}
{'-'*100}
{d_labitems.count()}
{'-'*100}
{d_labitems.isna().sum()}
{'-'*100}
{d_labitems.describe()}
{'-'*100}
''')

d_labitems:
(1650, 4)
----------------------------------------------------------------------------------------------------
itemid       int64
label       object
fluid       object
category    object
dtype: object
----------------------------------------------------------------------------------------------------
itemid      1650
label       1646
fluid       1650
category    1650
dtype: int64
----------------------------------------------------------------------------------------------------
itemid      0
label       4
fluid       0
category    0
dtype: int64
----------------------------------------------------------------------------------------------------
             itemid
count   1650.000000
mean   51734.312727
std      602.758734
min    50801.000000
25%    51227.250000
50%    51705.500000
75%    52147.750000
max    53190.000000
----------------------------------------------------------------------------------------------------



In [36]:
# filter only blood tests since Blood labs are more consistent across patients.
# for fluid, count in d_labitems['fluid'].value_counts().items():
#     print(fluid, count)
d_labitems_clean = d_labitems[d_labitems['fluid'] == 'Blood'].copy()
cols = ['itemid', 'label']
d_labitems_clean = d_labitems_clean[cols]

# Drop the missing label
d_labitems_clean = d_labitems_clean.dropna(subset=['label'])

# Preview cleaned data
print(f'''d_labitems_clean:
{d_labitems_clean.shape}
{'-'*100}
{d_labitems_clean.count()}
{'-'*100}
{d_labitems_clean.isna().sum()}
{'-'*100}
''')
d_labitems_clean.info()
d_labitems_clean.head()

d_labitems_clean:
(820, 2)
----------------------------------------------------------------------------------------------------
itemid    820
label     820
dtype: int64
----------------------------------------------------------------------------------------------------
itemid    0
label     0
dtype: int64
----------------------------------------------------------------------------------------------------

<class 'pandas.core.frame.DataFrame'>
Index: 820 entries, 0 to 1649
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   itemid  820 non-null    int64 
 1   label   820 non-null    object
dtypes: int64(1), object(1)
memory usage: 19.2+ KB


,itemid,label
0,50801,Alveolar-arterial Gradient
1,50802,Base Excess
2,50803,"Calculated Bicarbonate, Whole Blood"
3,50804,Calculated Total CO2
4,50805,Carboxyhemoglobin


### Merge tables


In [79]:
# Start from icustays and merge patients and admissions
df = icustays_clean.copy()
df = df.merge(patients_clean, on='subject_id', how='left')
df = df.merge(admissions_clean, on=['subject_id', 'hadm_id'], how='left')

# -------------------- CHART EVENTS (Vitals) --------------------

# Filter to vital signs only
vital_itemids = d_items_clean[d_items_clean['is_vital_sign'] == 1]['itemid'].unique()
chartevents_vital = chartevents_clean[chartevents_clean['itemid'].isin(vital_itemids)].copy()

# Merge label and ICU time for filtering
chartevents_vital = chartevents_vital.merge(d_items_clean[['itemid', 'label']], on='itemid', how='left')
chartevents_vital = chartevents_vital.merge(icustays_clean[['stay_id', 'intime', 'outtime']], on='stay_id', how='left')

# Filter chart events to those within ICU stay
chartevents_vital = chartevents_vital[
    (chartevents_vital['charttime'] >= chartevents_vital['intime']) &
    (chartevents_vital['charttime'] <= chartevents_vital['outtime'])
].copy()

# Aggregate by stay_id + itemid
chart_agg = (
    chartevents_vital
    .groupby(['stay_id', 'itemid'])['valuenum']
    .agg(['mean', 'std'])
    .reset_index()
)

# Pivot into wide format
chart_wide = chart_agg.pivot(index='stay_id', columns='itemid')
chart_wide.columns = [f'chart_{itemid}_{stat}' for itemid, stat in chart_wide.columns]
chart_wide = chart_wide.reset_index()

# Merge into main df
df = df.merge(chart_wide, on='stay_id', how='left')

# Handle chart missing values
for col in df.columns:
    if col.startswith("chart_") and ("_mean" in col or "_std" in col):
        df[col + "_missing"] = df[col].isna().astype(int)
        df[col] = df[col].fillna(-1)

# -------------------- LAB EVENTS (Blood Labs) --------------------

# Filter to lab itemids and merge with labels
lab_itemids = d_labitems_clean['itemid'].unique()
labevents_clean_blood = labevents_clean[labevents_clean['itemid'].isin(lab_itemids)].copy()
labevents_clean_blood = labevents_clean_blood.merge(d_labitems_clean[['itemid', 'label']], on='itemid', how='left')

# Merge in ICU admission times
labevents_clean_blood = labevents_clean_blood.merge(
    icustays_clean[['subject_id', 'hadm_id', 'intime', 'outtime']],
    on=['subject_id', 'hadm_id'],
    how='left'
)

# Filter labs to ICU window
labevents_clean_blood = labevents_clean_blood[
    (labevents_clean_blood['charttime'] >= labevents_clean_blood['intime']) &
    (labevents_clean_blood['charttime'] <= labevents_clean_blood['outtime'])
].copy()

# Aggregate by subject_id, hadm_id, and itemid
lab_agg = (
    labevents_clean_blood
    .groupby(['subject_id', 'hadm_id', 'itemid'])['valuenum']
    .agg(['mean', 'std'])
    .reset_index()
)

# Pivot into wide format
lab_wide = lab_agg.pivot(index=['subject_id', 'hadm_id'], columns='itemid')
lab_wide.columns = [f'lab_{itemid}_{stat}' for itemid, stat in lab_wide.columns]
lab_wide = lab_wide.reset_index()

# Merge into main df
df = df.merge(lab_wide, on=['subject_id', 'hadm_id'], how='left')

# Handle lab missing values
for col in df.columns:
    if col.startswith("lab_") and ("_mean" in col or "_std" in col):
        df[col + "_missing"] = df[col].isna().astype(int)
        df[col] = df[col].fillna(-1)

print(f'''df:
{df.shape}
{'-'*100}
{df.isna().sum()}
{'-'*100}
''')

df:
(94444, 44)
----------------------------------------------------------------------------------------------------
subject_id                       0
hadm_id                          0
stay_id                          0
intime                           0
outtime                          0
los                              0
gender                           0
anchor_age                       0
dod                          56485
dischtime                        0
hospital_expire_flag             0
race_grouped                     0
chart_mean_220045                0
chart_mean_220179                0
chart_mean_220180                0
chart_std_220045                 0
chart_std_220179                 0
chart_std_220180                 0
chart_mean_220045_missing        0
chart_mean_220179_missing        0
chart_mean_220180_missing        0
chart_std_220045_missing         0
chart_std_220179_missing         0
chart_std_220180_missing         0
lab_mean_50912                   0
lab_mean

In [80]:
df

,subject_id,hadm_id,stay_id,intime,outtime,los,gender,anchor_age,dod,dischtime,...,lab_mean_50912_missing,lab_mean_51006_missing,lab_mean_51221_missing,lab_mean_51222_missing,lab_mean_51265_missing,lab_std_50912_missing,lab_std_51006_missing,lab_std_51221_missing,lab_std_51222_missing,lab_std_51265_missing
0,10000032,29079034,39553978,2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266,0,52,2180-09-09,2180-07-25 17:55:00,...,0,0,1,1,1,1,1,1,1,1
1,10000690,25860671,37081114,2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252,0,86,2152-01-30,2150-11-12 13:45:00,...,0,0,0,0,0,0,0,0,0,0
2,10000980,26913865,39765666,2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535,0,73,2193-08-26,2189-07-03 03:00:00,...,1,1,1,1,1,1,1,1,1,1
3,10001217,24597018,37067082,2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032,0,55,NaT,2157-11-25 18:00:00,...,0,0,0,0,0,1,1,1,1,1
4,10001217,27703517,34592300,2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113,0,55,NaT,2157-12-24 14:55:00,...,0,0,0,0,0,1,1,1,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94439,19999442,26785317,32336619,2148-11-19 14:23:43,2148-11-26 13:12:15,6.950370,1,41,NaT,2148-12-04 16:25:00,...,0,0,0,0,0,0,0,0,0,0
94440,19999625,25304202,31070865,2139-10-10 19:18:00,2139-10-11 18:21:28,0.960741,1,81,NaT,2139-10-16 03:30:00,...,0,0,0,0,0,0,0,0,1,1
94441,19999828,25744818,36075953,2149-01-08 18:12:00,2149-01-10 13:11:02,1.790995,0,46,NaT,2149-01-18 17:00:00,...,0,0,0,0,0,0,0,0,0,0
94442,19999840,21033226,38978960,2164-09-12 09:26:28,2164-09-17 16:35:15,5.297766,1,58,2164-09-17,2164-09-17 13:42:00,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
df_data = df.copy()

# One-hot encode race_grouped
race_dummies = pd.get_dummies(df_data['race_grouped'], prefix='race', drop_first=True)
# Add to df_data and drop original
df_data = pd.concat([df_data.drop(columns=['race_grouped']), race_dummies], axis=1)

# ICU mortality label: whether the patient died during ICU stay
df_data['icu_mortality'] = df_data.apply(lambda row: int(pd.notna(row['dod']) and row['dod'] <= row['outtime']), axis=1)

# after computing icu_mortality, you do not need dod anymore
cols_to_drop = [
    'intime', 'outtime', 'dod', 'dischtime',  # time-based columns not used directly
    'subject_id', 'hadm_id', 'stay_id',       # identifiers
    'hospital_expire_flag'                   # patient died during the entire hospitalization
]
df_data = df_data.drop(columns=cols_to_drop)

print(f'''df:
{df_data.shape}
{'-'*100}
{df_data.isna().sum()}
{'-'*100}
''')


df:
(94444, 43)
----------------------------------------------------------------------------------------------------
los                          0
gender                       0
anchor_age                   0
chart_mean_220045            0
chart_mean_220179            0
chart_mean_220180            0
chart_std_220045             0
chart_std_220179             0
chart_std_220180             0
chart_mean_220045_missing    0
chart_mean_220179_missing    0
chart_mean_220180_missing    0
chart_std_220045_missing     0
chart_std_220179_missing     0
chart_std_220180_missing     0
lab_mean_50912               0
lab_mean_51006               0
lab_mean_51221               0
lab_mean_51222               0
lab_mean_51265               0
lab_std_50912                0
lab_std_51006                0
lab_std_51221                0
lab_std_51222                0
lab_std_51265                0
lab_mean_50912_missing       0
lab_mean_51006_missing       0
lab_mean_51221_missing       0
lab_mean_51222_

In [82]:
df_data

,los,gender,anchor_age,chart_mean_220045,chart_mean_220179,chart_mean_220180,chart_std_220045,chart_std_220179,chart_std_220180,chart_mean_220045_missing,...,lab_std_51222_missing,lab_std_51265_missing,race_ASIAN,race_BLACK,race_HISPANIC/LATINO,race_OTHER,race_PACIFIC ISLANDER,race_UNKNOWN/DECLINED,race_WHITE,icu_mortality
0,0.410266,0,52,96.500000,88.900000,54.100000,4.196559,4.629615,5.646041,0,...,1,1,False,False,False,False,False,False,True,0
1,3.893252,0,86,84.072917,122.893617,60.361702,15.712713,19.736146,16.086360,0,...,0,0,False,False,False,False,False,False,True,0
2,0.497535,0,73,73.636364,142.454545,83.272727,3.585324,10.162319,16.006817,0,...,1,1,False,True,False,False,False,False,False,0
3,1.118032,0,55,93.296296,136.296296,81.333333,7.363187,8.511598,10.528350,0,...,1,1,False,False,False,False,False,False,True,0
4,0.948113,0,55,79.600000,115.869565,73.478261,7.697402,19.038966,11.143158,0,...,1,1,False,False,False,False,False,False,True,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94439,6.950370,1,41,59.453039,140.431818,79.181818,16.549060,7.647974,6.111956,0,...,0,0,False,False,False,False,False,False,True,0
94440,0.960741,1,81,71.000000,114.227273,57.272727,8.551633,12.731408,7.759368,0,...,1,1,False,False,False,False,False,False,True,0
94441,1.790995,0,46,94.756098,108.142857,75.571429,8.865045,9.470291,8.364101,0,...,0,0,False,False,False,False,False,False,True,0
94442,5.297766,1,58,73.479042,103.883436,53.509202,10.910279,15.003865,7.757408,0,...,0,0,False,False,False,False,False,False,True,1


In [91]:
df_sensitive = df.copy()
df_sensitive = df_sensitive.drop(columns=['gender', 'anchor_age', 'race_grouped'])
# ICU mortality label: whether the patient died during ICU stay
df_sensitive['icu_mortality'] = df_sensitive.apply(lambda row: int(pd.notna(row['dod']) and row['dod'] <= row['outtime']), axis=1)

# after computing icu_mortality, you do not need dod anymore
cols_to_drop = [
    'intime', 'outtime', 'dod', 'dischtime',  # time-based columns not used directly
    'subject_id', 'hadm_id', 'stay_id',       # identifiers
    'hospital_expire_flag'                   # patient died during the entire hospitalization
]
df_sensitive = df_sensitive.drop(columns=cols_to_drop)

print(f'''df:
{df_sensitive.shape}
{'-'*100}
{df_sensitive.isna().sum()}
{'-'*100}
''')

df:
(94444, 34)
----------------------------------------------------------------------------------------------------
los                          0
chart_mean_220045            0
chart_mean_220179            0
chart_mean_220180            0
chart_std_220045             0
chart_std_220179             0
chart_std_220180             0
chart_mean_220045_missing    0
chart_mean_220179_missing    0
chart_mean_220180_missing    0
chart_std_220045_missing     0
chart_std_220179_missing     0
chart_std_220180_missing     0
lab_mean_50912               0
lab_mean_51006               0
lab_mean_51221               0
lab_mean_51222               0
lab_mean_51265               0
lab_std_50912                0
lab_std_51006                0
lab_std_51221                0
lab_std_51222                0
lab_std_51265                0
lab_mean_50912_missing       0
lab_mean_51006_missing       0
lab_mean_51221_missing       0
lab_mean_51222_missing       0
lab_mean_51265_missing       0
lab_std_50912_m

In [92]:
df_sensitive

,los,chart_mean_220045,chart_mean_220179,chart_mean_220180,chart_std_220045,chart_std_220179,chart_std_220180,chart_mean_220045_missing,chart_mean_220179_missing,chart_mean_220180_missing,...,lab_mean_51006_missing,lab_mean_51221_missing,lab_mean_51222_missing,lab_mean_51265_missing,lab_std_50912_missing,lab_std_51006_missing,lab_std_51221_missing,lab_std_51222_missing,lab_std_51265_missing,icu_mortality
0,0.410266,96.500000,88.900000,54.100000,4.196559,4.629615,5.646041,0,0,0,...,0,1,1,1,1,1,1,1,1,0
1,3.893252,84.072917,122.893617,60.361702,15.712713,19.736146,16.086360,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0.497535,73.636364,142.454545,83.272727,3.585324,10.162319,16.006817,0,0,0,...,1,1,1,1,1,1,1,1,1,0
3,1.118032,93.296296,136.296296,81.333333,7.363187,8.511598,10.528350,0,0,0,...,0,0,0,0,1,1,1,1,1,0
4,0.948113,79.600000,115.869565,73.478261,7.697402,19.038966,11.143158,0,0,0,...,0,0,0,0,1,1,1,1,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94439,6.950370,59.453039,140.431818,79.181818,16.549060,7.647974,6.111956,0,0,0,...,0,0,0,0,0,0,0,0,0,0
94440,0.960741,71.000000,114.227273,57.272727,8.551633,12.731408,7.759368,0,0,0,...,0,0,0,0,0,0,0,1,1,0
94441,1.790995,94.756098,108.142857,75.571429,8.865045,9.470291,8.364101,0,0,0,...,0,0,0,0,0,0,0,0,0,0
94442,5.297766,73.479042,103.883436,53.509202,10.910279,15.003865,7.757408,0,0,0,...,0,0,0,0,0,0,0,0,0,1


### logistic regression

In [93]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score

X = df_data.drop(columns='icu_mortality')
y = df_data['icu_mortality']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, stratify=y, random_state=35)

model = LogisticRegression(max_iter=1000, solver='liblinear')
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))


              precision    recall  f1-score   support

           0       0.93      0.99      0.96     26008
           1       0.64      0.16      0.25      2326

    accuracy                           0.92     28334
   macro avg       0.78      0.57      0.61     28334
weighted avg       0.91      0.92      0.90     28334

ROC AUC: 0.8373431893301961
